# Refusal Style Trainer SFT

This notebook trains a LoRA adapter on a small base model (Qwen2.5-0.5B-Instruct) to adopt a refusal style for harmful prompts, while remaining helpful for benign prompts.

First, we install the necessary libraries.

In [ ]:
!pip install -q transformers peft trl torch datasets accelerate bitsandbytes

## 1. Upload Data
Please ensure `data/train.jsonl` and `data/val.jsonl` are uploaded to your Colab environment in a `data/` folder.

In [ ]:
import os
os.makedirs("data", exist_ok=True)
os.makedirs("models/sft-lora", exist_ok=True)
# You can upload your files into the 'data' folder using the files pane on the left.

## 2. Run Training
The code below uses `trl` and `peft` to fine-tune the model.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# Configuration
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
output_dir = "./models/sft-lora"
data_path = "./data/train.jsonl"
val_path = "./data/val.jsonl"

# Load tokenizers
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Load datasets
train_dataset = load_dataset('json', data_files=data_path, split='train')
val_dataset = load_dataset('json', data_files=val_path, split='train')

def format_prompts(example):
    messages = example['messages']
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_dataset = train_dataset.map(format_prompts)
val_dataset = val_dataset.map(format_prompts)

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="adamw_torch",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    logging_steps=10,
    num_train_epochs=3,
    max_steps=-1,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",
    evaluation_strategy="epoch",
)

response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
    data_collator=collator
)

trainer.train()

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
